In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from IPython.display import display

sys.path.append(os.path.abspath('..'))

# Importamos nuestras clases (MLOps)
from src.transformadores import LimpiadorFormatosKaggle, CalculadorDistancia

print("=====================================================================")
print(" FASE 4: CARGA DE DATOS Y SEPARACIÓN DE X e Y")
print("=====================================================================\n")

# Cargar los datos estratificados
data_path = os.path.join('..', 'data', 'processed', 'train_processed.csv')
df_train = pd.read_csv(data_path)

# Separar variables predictoras (X) de la variable objetivo (y)
X_train = df_train.drop(columns=['Time_taken(min)'])


y_train = df_train['Time_taken(min)'].astype(str).str.extract(r'(\d+\.?\d*)')[0].astype(float)

print(f"Variables predictoras (X_train) cargadas: {X_train.shape}")
print(f"Etiqueta objetivo (y_train) cargada: {y_train.shape}")

 FASE 4: CARGA DE DATOS Y SEPARACIÓN DE X e Y

✓ Variables predictoras (X_train) cargadas: (36474, 19)
✓ Etiqueta objetivo (y_train) cargada: (36474,)


In [ ]:
from sklearn.pipeline import Pipeline

print("=====================================================================")
print(" FASE 4: APLICACIÓN DE FEATURE ENGINEERING (HAVERSINE Y REGEX)")
print("=====================================================================\n")

# Construir la primera fase de la pipeline
pipeline_base = Pipeline([
    ('limpiador', LimpiadorFormatosKaggle()),
    ('calculador_distancia', CalculadorDistancia())
])

# Pasar los datos crudos por la pipeline
X_train_base = pipeline_base.fit_transform(X_train)

print("Transformación exitosa")
print("Nueva variable: 'Distancia_km'.\n")
display(X_train_base[['Delivery_person_Age', 'Road_traffic_density', 'Distancia_km']].head())

 FASE 4: APLICACIÓN DE FEATURE ENGINEERING (HAVERSINE Y REGEX)

¡Transformación exitosa! Observa cómo desaparecieron las 4 coordenadas
y apareció nuestra nueva súper-variable: 'Distancia_km'.



,Delivery_person_Age,Road_traffic_density,Distancia_km
0,24.0,High,3.018911
1,38.0,Low,1.498138
2,37.0,Medium,4.589099
3,34.0,High,3.040542
4,30.0,Medium,20.182483


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

print("=====================================================================")
print(" FASE 4: IMPUTACIÓN DE NULOS Y PIPELINE")
print("=====================================================================\n")

# 1. Definimos qué columnas son números y cuáles son categorías
# IMPORTANTE: Ya usamos 'Distancia_km'
num_cols = ['Delivery_person_Age', 'Delivery_person_Ratings', 'multiple_deliveries', 'Vehicle_condition', 'Distancia_km']
cat_cols = ['Weatherconditions', 'Road_traffic_density', 'Type_of_vehicle', 'Festival', 'City']

# 2. Sub-pipeline para números: Mediana + Escalado Estándar
num_pipeline = Pipeline([
    ('imputador', SimpleImputer(strategy='median')),
    ('escalador', StandardScaler())
])

# 3. Sub-pipeline para categorías: Moda + One-Hot Encoding
cat_pipeline = Pipeline([
    ('imputador', SimpleImputer(strategy='most_frequent')),
    ('codificador', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 4. Unimos todo con ColumnTransformer
preprocesador_final = ColumnTransformer([
    ('numericos', num_pipeline, num_cols),
    ('categoricos', cat_pipeline, cat_cols)
])

# 5. PIPELINE DEFINITIVO (Nuestras clases + Las clases de Scikit-Learn)
pipeline_completo = Pipeline([
    ('preparacion_base', pipeline_base),
    ('preprocesamiento_final', preprocesador_final)
])

# 6. Ejecutamos todo
X_train_preparado = pipeline_completo.fit_transform(X_train)

print("Matriz final lista para Machine Learning.")
print(f"Dimensiones finales: {X_train_preparado.shape}")

 FASE 4: IMPUTACIÓN DE NULOS Y PIPELINE

✓ Matriz final lista para Machine Learning.
Dimensiones finales: (36474, 25)
Tus datos pasaron de ser un desastre de texto a una matriz 100% numérica.
